# **Pythonプログラミング演習 2026年度 実践課題**

## **課題概要**

本課題では，**「オセロをプレイするAI」** を自作します．

独自の `Player` クラスを実装し，[Webシステム](https://othellopy.com) より提出してください．

## **課題提出**

**【中間提出】**

**提出物**：
* 実装したオセロAI．[Webシステム](https://othellopy.com) より提出

**締切**：**08/03**（第13・14回講義日） **23:59**\
**採点**：提出点のみ

**【最終提出】**

**提出物**：
* 実装したオセロAI．[Webシステム](https://othellopy.com) より提出
* 実践課題レポート（後日指示）

**締切**：**09/30** **23:59**\
（早期提出：08/31）

**採点**：
* 提出点
* 性能点：提出したオセロAIの性能に応じて採点
    * 「秀」の目安：中級レベルのCPU `IntermediatePlayer` に確実に勝てる性能であること
    * 最終提出の性能に応じて加点あり
* 実践課題レポートで工夫点を詳しく説明している場合加点あり

## **問い合わせ**

* よくある質問は [FAQページ](https://othellopy.com/faq) にまとめています．

不明点は，以下の方法でお問い合わせください．
* 授業中，教員もしくは TA に質問する（現地・オンライン）
* [TAのメーリングリスト](https://mail.google.com/mail/?view=cm&fs=1&to=python_pp_ta@ml.naist.ac.jp) `python_pp_ta@ml.naist.ac.jp` に質問内容をメールする

---
## **ソースコード本体**

### **必要なパッケージのインストール**

本課題では Python でオセロを実装するためのライブラリ `othellopy` を利用します．

In [ ]:
!pip install -U othellopy

: 

### **必要なライブラリのインポート**

In [11]:
from othellopy.core import Board, Cell, Move
# Board: オセロ盤面を表す型．8 x 8 の2次元配列．
# Cell: 各マスの状態を表す値．EMPTY（0), BLACK (1), WHITE (2)．
# Move: 石を置く位置を表す型．(row, col) のタプル．

from othellopy.players import BasePlayer
# BasePlayer: 自作プレイヤーの基底クラス．get_moves などの補助メソッドを提供.

### **<font color="red">【提出対象】自作AI `MyPlayer` の作成</font>**

以下の class `MyPlayer` が制作・提出対象です．\
自分なりにアレンジして強い「オセロAI」を作成してください．

#### **サンプル**

```
class MyPlayer(BasePlayer):
    def next_move(self, board: Board) -> Move:
        moves = self.get_moves(board)
        return moves[0]
```

#### **実装時の制約**

* 利用できるライブラリは標準ライブラリのみ（`random`, `math` 等）
* 外部ライブラリ（`numpy`，`scipy` 等）利用NG．
* 外部データ参照NG（`requests`，`os` 等）
* `next_move()` の実行時間制限は2秒（重すぎる処理はNG）
* 生成AI利用可（出力を鵜呑みにせず，有効に活用してください）

### **解説**

* `next_move()`
  * 現在の盤面を受け取り，次に打つべき１手を決定する．
  * 返り値：次に打つ手の座標（例：`(0, 0)`）
* `board: Board`
  * 盤面を管理する二次元配列
  * 左上が `board[0][0]`
  * 例：`board[3][5]` = 上から4段目，左から6列目
  * `0` = 空白，`1` = 黒，`2` = 白
* `self.get_moves(board)`
  * 現在の盤面での有効手を一覧で取得する．
  * 例：`[(2, 3), (3, 2), (4, 5), (5, 4)]`

### **提出方法**

[Webシステム](https://othellopy.com) より以下のコードセルで作成したオセロAIをコピー＆ペーストして提出

---

**<font color="red">↓ 以下のコードセルを実装・提出</font>**

In [ ]:
from othellopy.core import Board, Cell, Move
from othellopy.players import BasePlayer


# 提出時は，本コードセルをコピーして提出する
class MyPlayer(BasePlayer):
    # 画像の「マスの評価」。board[row][col] と同じ 0 始まりの行・列で参照する。
    POSITION_WEIGHTS = (
        (2714, 147, 69, -18, -18, 69, 147, 2714),
        (147, -577, -186, -153, -153, -186, -577, 147),
        (69, -186, -379, -122, -122, -379, -186, 69),
        (-18, -153, -122, -169, -169, -122, -153, -18),
        (-18, -153, -122, -169, -169, -122, -153, -18),
        (69, -186, -379, -122, -122, -379, -186, 69),
        (147, -577, -186, -153, -153, -186, -577, 147),
        (2714, 147, 69, -18, -18, 69, 147, 2714),
    )
    BOARD_INDEXES = range(8)
    SEARCH_DEPTH = 7
    DIRECTIONS = (
        (-1, -1), (-1, 0), (-1, 1),
        (0, -1),           (0, 1),
        (1, -1),  (1, 0),  (1, 1),
    )

    def next_move(self, board: Board) -> Move:
        moves = self.get_moves(board)
        if not moves:
            return None

        best_move = moves[0]
        best_score = float("-inf")
        alpha = float("-inf")
        beta = float("inf")

        for move in moves:
            next_board = self._apply_move(board, move, self.color)
            score = -self._negascout(
                next_board,
                depth=self.SEARCH_DEPTH - 1,
                current_color=self._opponent_of(self.color),
                alpha=-beta,
                beta=-alpha,
            )
            if score > best_score:
                best_score = score
                best_move = move
            alpha = max(alpha, best_score)

        return best_move

    def _negascout(
        self,
        board: Board,
        depth: int,
        current_color: Cell,
        alpha: float,
        beta: float,
    ) -> int:
        if depth == 0:
            return self._evaluate_for_color(board, current_color)

        moves = self._legal_moves(board, current_color)
        next_color = self._opponent_of(current_color)

        # 打てる手がない場合はパス。両者とも打てなければその盤面を評価する。
        if not moves:
            if not self._legal_moves(board, next_color):
                return self._evaluate_for_color(board, current_color)
            return -self._negascout(board, depth, next_color, -beta, -alpha)

        search_window = beta
        best_score = float("-inf")

        for index, move in enumerate(moves):
            next_board = self._apply_move(board, move, current_color)
            score = -self._negascout(
                next_board, depth - 1, next_color, -search_window, -alpha
            )

            if alpha < score < beta and index > 0 and depth > 1:
                score = -self._negascout(
                    next_board, depth - 1, next_color, -beta, -score
                )

            best_score = max(best_score, score)
            alpha = max(alpha, score)
            if alpha >= beta:
                break
            search_window = alpha + 1

        return best_score

    def _evaluate_for_color(self, board: Board, color: Cell) -> int:
        score = self._evaluate_board(board)
        if color == self.color:
            return score
        return -score

    def _evaluate_board(self, board: Board) -> int:
        opponent_color = self._opponent_of(self.color)
        score = 0

        for row in self.BOARD_INDEXES:
            for col in self.BOARD_INDEXES:
                cell = board[row][col]
                weight = self.POSITION_WEIGHTS[row][col]
                if cell == self.color:
                    score += weight
                elif cell == opponent_color:
                    score -= weight

        return score

    def _legal_moves(self, board: Board, color: Cell) -> list[Move]:
        moves = []
        for row in self.BOARD_INDEXES:
            for col in self.BOARD_INDEXES:
                if board[row][col] == Cell.EMPTY and self._flips_for_color(
                    board, row, col, color
                ):
                    moves.append((row, col))
        return moves

    def _apply_move(self, board: Board, move: Move, color: Cell) -> Board:
        row, col = move
        next_board = [board_row[:] for board_row in board]
        next_board[row][col] = color

        for flip_row, flip_col in self._flips_for_color(board, row, col, color):
            next_board[flip_row][flip_col] = color

        return next_board

    def _flips_for_color(
        self, board: Board, row: int, col: int, color: Cell
    ) -> list[Move]:
        if board[row][col] != Cell.EMPTY:
            return []

        opponent_color = self._opponent_of(color)
        flips = []

        for delta_row, delta_col in self.DIRECTIONS:
            direction_flips = []
            current_row = row + delta_row
            current_col = col + delta_col

            while (
                0 <= current_row < 8
                and 0 <= current_col < 8
                and board[current_row][current_col] == opponent_color
            ):
                direction_flips.append((current_row, current_col))
                current_row += delta_row
                current_col += delta_col

            if (
                direction_flips
                and 0 <= current_row < 8
                and 0 <= current_col < 8
                and board[current_row][current_col] == color
            ):
                flips.extend(direction_flips)

        return flips

    def _opponent_of(self, color: Cell) -> Cell:
        return Cell.WHITE if color == Cell.BLACK else Cell.BLACK


強化学習を考えたコード↓


**<font color="red">↑ 以上のコードセルを実装・提出</font>**

---

### **相手AIとの対戦**

実際に対戦してみましょう．\
ライブラリ `othellopy` では，3レベルのAIを用意しています．

* `BeginnerPlayer`：初級．ランダムに置くだけの戦略
* `IntermediatePlayer`：中級．置くと有利になる場所に優先して置く戦略
* `AdvancedPlayer`：上級．数手先の盤面まで予測する戦略

In [14]:
from othellopy.game import OthelloGame
# OthelloGame: 試合の管理．2つのプレイヤークラスを渡し試合を行う
from othellopy.players import BeginnerPlayer, IntermediatePlayer, AdvancedPlayer

game = OthelloGame(
    black_player=MyPlayer,      # この行を変更して黒石のプレイヤーを変更
    white_player=BeginnerPlayer # この行を変更して白石プレイヤーを変更
)

result = game.play()

### **結果の表示**

In [15]:
from othellopy.board import display_board
# display_board: 盤面をグラフィカルに表示する関数

print("Winner:", result.winner_name)
print("Black:", result.black_score)
print("White", result.white_score)
display_board(result.board)

Winner: BLACK
Black: 37
White 27


,0,1,2,3,4,5,6,7
0,⚫️,⚫️,⚪️,⚪️,⚪️,⚪️,⚪️,⚪️
1,⚫️,⚫️,⚫️,⚪️,⚫️,⚪️,⚪️,⚫️
2,⚫️,⚫️,⚪️,⚫️,⚪️,⚪️,⚪️,⚫️
3,⚫️,⚪️,⚫️,⚪️,⚪️,⚪️,⚫️,⚫️
4,⚫️,⚫️,⚪️,⚪️,⚪️,⚫️,⚫️,⚫️
5,⚫️,⚪️,⚫️,⚪️,⚫️,⚪️,⚫️,⚫️
6,⚪️,⚪️,⚪️,⚫️,⚫️,⚫️,⚪️,⚫️
7,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️


1手ずつ詳細を確認することもできます．

In [16]:
for i, turn in enumerate(result.turns):
    print(f"Turn {i}: {turn.color.name}")

    if turn.move is None:
        print("Move: pass")
    else:
        row, col = turn.move
        print(f"Move: {row}{col}")

    print(f"Valid moves: {turn.valid_moves}")
    print(f"Score: BLACK {turn.black_score} - WHITE {turn.white_score}")
    display_board(turn.board)
    print()

print("Winner:", result.winner_name)

Turn 0: BLACK
Move: 23
Valid moves: [(2, 3), (3, 2), (4, 5), (5, 4)]
Score: BLACK 4 - WHITE 1


,0,1,2,3,4,5,6,7
0,.,.,.,.,.,.,.,.
1,.,.,.,.,.,.,.,.
2,.,.,.,⚫️,.,.,.,.
3,.,.,.,⚫️,⚫️,.,.,.
4,.,.,.,⚫️,⚪️,.,.,.
5,.,.,.,.,.,.,.,.
6,.,.,.,.,.,.,.,.
7,.,.,.,.,.,.,.,.



Turn 1: WHITE
Move: 24
Valid moves: [(2, 2), (2, 4), (4, 2)]
Score: BLACK 3 - WHITE 3


,0,1,2,3,4,5,6,7
0,.,.,.,.,.,.,.,.
1,.,.,.,.,.,.,.,.
2,.,.,.,⚫️,⚪️,.,.,.
3,.,.,.,⚫️,⚪️,.,.,.
4,.,.,.,⚫️,⚪️,.,.,.
5,.,.,.,.,.,.,.,.
6,.,.,.,.,.,.,.,.
7,.,.,.,.,.,.,.,.



Turn 2: BLACK
Move: 35
Valid moves: [(1, 5), (2, 5), (3, 5), (4, 5), (5, 5)]
Score: BLACK 5 - WHITE 2


...


Turn 60: BLACK
Move: pass
Valid moves: []
Score: BLACK 37 - WHITE 27


,0,1,2,3,4,5,6,7
0,⚫️,⚫️,⚪️,⚪️,⚪️,⚪️,⚪️,⚪️
1,⚫️,⚫️,⚫️,⚪️,⚫️,⚪️,⚪️,⚫️
2,⚫️,⚫️,⚪️,⚫️,⚪️,⚪️,⚪️,⚫️
3,⚫️,⚪️,⚫️,⚪️,⚪️,⚪️,⚫️,⚫️
4,⚫️,⚫️,⚪️,⚪️,⚪️,⚫️,⚫️,⚫️
5,⚫️,⚪️,⚫️,⚪️,⚫️,⚪️,⚫️,⚫️
6,⚪️,⚪️,⚪️,⚫️,⚫️,⚫️,⚪️,⚫️
7,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️



Turn 61: WHITE
Move: pass
Valid moves: []
Score: BLACK 37 - WHITE 27


,0,1,2,3,4,5,6,7
0,⚫️,⚫️,⚪️,⚪️,⚪️,⚪️,⚪️,⚪️
1,⚫️,⚫️,⚫️,⚪️,⚫️,⚪️,⚪️,⚫️
2,⚫️,⚫️,⚪️,⚫️,⚪️,⚪️,⚪️,⚫️
3,⚫️,⚪️,⚫️,⚪️,⚪️,⚪️,⚫️,⚫️
4,⚫️,⚫️,⚪️,⚪️,⚪️,⚫️,⚫️,⚫️
5,⚫️,⚪️,⚫️,⚪️,⚫️,⚪️,⚫️,⚫️
6,⚪️,⚪️,⚪️,⚫️,⚫️,⚫️,⚪️,⚫️
7,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️,⚫️



Winner: BLACK


### **`MyPlayer` の提出チェック**

自作した `MyPlayer` に不具合がないかを確認するためのテストを用意しています．

提出前のチェックにご利用ください．


In [17]:
from othellopy.validation import test_player, test_player_detail
# test_player: 作成したプレイヤークラスの簡易動作チェック
# test_player_detail: 作成したプレイヤークラスの簡易動作チェック（詳細情報付き）

if test_player(MyPlayer):
    print("テストを PASS しました．")
else:
    result = test_player_detail(MyPlayer)
    for issue in result.errors:
        print(issue.code, issue.message)

テストを PASS しました．
